In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.spatial import KDTree
import ipywidgets as widgets
from IPython.display import display
import anndata as ad

def interactive_feature_graph(adata, pixel_size=60, top_n=50):
    """
    Interactive visualization of top-intensity pixel graphs for all features in an AnnData object.

    Parameters
    ----------
    adata : AnnData
        AnnData object containing intensity matrix (adata.X)
        and spatial coordinates in adata.obs[['x', 'y']].
    pixel_size : float, optional
        Pixel size in micrometers (default = 60).
    top_n : int, optional
        Number of top-intensity pixels to select (default = 50).
    """

    features = adata.var_names.tolist()

    # Dropdown widget for feature selection
    feature_dropdown = widgets.Dropdown(
        options=features,
        description='Feature:',
        layout=widgets.Layout(width='50%')
    )

    output = widgets.Output()

    def plot_feature_graph(feature):
        output.clear_output(wait=True)
        with output:
            x = adata.obs["x"].values
            y = adata.obs["y"].values
            intensities = np.array(adata[:, feature].X).flatten()

            # Find top N pixels
            top_indices = np.argsort(intensities)[-top_n:]
            coords = np.column_stack([x[top_indices], y[top_indices]])
            top_values = intensities[top_indices]

            # Build KDTree for neighbors
            tree = KDTree(coords)
            radius = top_n * pixel_size
            pairs = tree.query_pairs(radius)

            # Build edge list
            edges = []
            for i, j in pairs:
                dist = np.linalg.norm(coords[i] - coords[j])
                edges.append((i, j, dist))

            # Compute node degree (number of neighbors)
            degrees = np.zeros(len(coords))
            for i, j, _ in edges:
                degrees[i] += 1
                degrees[j] += 1

            # Create Plotly figure
            fig = go.Figure()

            # Add edges (lines)
            for i, j, dist in edges:
                fig.add_trace(go.Scatter(
                    x=[coords[i, 0], coords[j, 0]],
                    y=[coords[i, 1], coords[j, 1]],
                    mode='lines',
                    line=dict(width=max(0.5, 4 - dist / radius * 3), color='gray'),
                    hoverinfo='none'
                ))

            # Add nodes (points)
            fig.add_trace(go.Scatter(
                x=coords[:, 0],
                y=coords[:, 1],
                mode='markers',
                marker=dict(
                    size=5 + 3 * degrees / np.max(degrees) if np.max(degrees) > 0 else 5,
                    color=top_values,
                    colorscale='Viridis',
                    showscale=True,
                    colorbar=dict(title="Intensity")
                ),
                text=[f"Intensity: {v:.2f}<br>Degree: {int(d)}" for v, d in zip(top_values, degrees)],
                hoverinfo='text',
                name='Pixels'
            ))

            fig.update_layout(
                title=f"Feature: {feature} — Top {top_n} Pixels (Edges weighted by distance)",
                xaxis=dict(title='X (μm)', scaleanchor='y', scaleratio=1),
                yaxis=dict(title='Y (μm)'),
                height=700,
                template='plotly_white'
            )

            fig.show()

    # Link dropdown to plotting function
    widgets.interactive_output(plot_feature_graph, {'feature': feature_dropdown})

    # Display controls + output
    display(widgets.VBox([feature_dropdown, output]))


In [2]:
import anndata as ad
adata = ad.read_h5ad("/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/lipids_common/msi_aad_1_20.h5ad")

In [ ]:
interactive_feature_graph(adata, pixel_size=60, top_n=10)


In [7]:
import numpy as np
import plotly.graph_objects as go
from scipy.spatial import KDTree
import ipywidgets as widgets
from IPython.display import display
import anndata as ad

def interactive_feature_graph_with_heatmap(adata, pixel_size=60, top_n=50):
    """
    Interactive visualization of top-intensity pixel graphs with heatmap background.

    Parameters
    ----------
    adata : AnnData
        AnnData object containing intensity matrix (adata.X)
        and spatial coordinates in adata.obs[['x', 'y']].
    pixel_size : float, optional
        Pixel size in micrometers (default = 60).
    top_n : int, optional
        Number of top-intensity pixels to select (default = 50).
    """

    features = adata.var_names.tolist()

    feature_dropdown = widgets.Dropdown(
        options=features,
        description='Feature:',
        layout=widgets.Layout(width='50%')
    )

    output = widgets.Output()

    def plot_feature_graph(feature):
        output.clear_output(wait=True)
        with output:
            x = adata.obs["x"].values
            y = adata.obs["y"].values
            intensities = np.array(adata[:, feature].X).flatten()

            # Top N pixels
            top_indices = np.argsort(intensities)[-top_n:]
            coords = np.column_stack([x[top_indices], y[top_indices]])
            top_values = intensities[top_indices]

            # KDTree for neighbors
            tree = KDTree(coords)
            radius = top_n * pixel_size
            pairs = tree.query_pairs(radius)

            edges = []
            for i, j in pairs:
                dist = np.linalg.norm(coords[i] - coords[j])
                edges.append((i, j, dist))

            degrees = np.zeros(len(coords))
            for i, j, _ in edges:
                degrees[i] += 1
                degrees[j] += 1

            # Build heatmap grid
            xi = np.unique(x)
            yi = np.unique(y)
            xi.sort()
            yi.sort()
            heatmap_matrix = np.full((len(yi), len(xi)), np.nan)

            x_idx = {val:i for i,val in enumerate(xi)}
            y_idx = {val:i for i,val in enumerate(yi)}
            for xi_val, yi_val, val in zip(x, y, intensities):
                heatmap_matrix[y_idx[yi_val], x_idx[xi_val]] = val

            # Plot
            fig = go.Figure()

            # Heatmap background
            fig.add_trace(go.Heatmap(
                z=heatmap_matrix,
                x=xi,
                y=yi,
                colorscale='Viridis',
                zmin=np.nanmin(intensities),
                zmax=np.nanmax(intensities),
                showscale=False,
                opacity=0.6
            ))

            # Edges
            for i, j, dist in edges:
                fig.add_trace(go.Scatter(
                    x=[coords[i,0], coords[j,0]],
                    y=[coords[i,1], coords[j,1]],
                    mode='lines',
                    line=dict(width=max(0.5, 4 - dist / radius * 3), color='gray'),
                    hoverinfo='none'
                ))

            # Nodes
            fig.add_trace(go.Scatter(
                x=coords[:,0],
                y=coords[:,1],
                mode='markers',
                marker=dict(
                    size=5 + 3 * degrees / np.max(degrees) if np.max(degrees) > 0 else 5,
                    color=top_values,
                    colorscale='Viridis',
                    showscale=True,
                    colorbar=dict(title="Intensity")
                ),
                text=[f"Intensity: {v:.4f}<br>Degree: {int(d)}" for v, d in zip(top_values, degrees)],
                hoverinfo='text',
                name='Top pixels'
            ))

            fig.update_layout(
                title=f"Feature: {feature} — Top {top_n} Pixels with Heatmap",
                xaxis=dict(title='X (μm)', scaleanchor='y', scaleratio=1),
                yaxis=dict(title='Y (μm)'),
                height=700,
                template='plotly_white'
            )

            fig.show()

    widgets.interactive_output(plot_feature_graph, {'feature': feature_dropdown})
    display(widgets.VBox([feature_dropdown, output]))


In [ ]:
interactive_feature_graph_with_heatmap(adata, pixel_size=60, top_n=50)


In [12]:
import numpy as np
import plotly.graph_objects as go
from scipy.spatial import KDTree
import ipywidgets as widgets
from IPython.display import display
import anndata as ad

def interactive_feature_graph_with_heatmap(adata, pixel_size=60, top_n=50, neighbor_distance=300):
    """
    Interactive visualization of top-intensity pixel graphs with heatmap background.
    Flipped horizontally and vertically.
    
    Parameters
    ----------
    adata : AnnData
        AnnData object containing intensity matrix (adata.X)
        and spatial coordinates in adata.obs[['x', 'y']].
    pixel_size : float, optional
        Pixel size in micrometers (default = 60).
    top_n : int, optional
        Number of top-intensity pixels to select (default = 50).
    neighbor_distance : float, optional
        Distance threshold in μm to connect neighbor pixels (default = 300).
    """

    features = adata.var_names.tolist()

    feature_dropdown = widgets.Dropdown(
        options=features,
        description='Feature:',
        layout=widgets.Layout(width='50%')
    )

    neighbor_slider = widgets.IntSlider(
        value=neighbor_distance,
        min=10,
        max=1000,
        step=10,
        description='Neighbor Dist (μm):',
        continuous_update=False,
        layout=widgets.Layout(width='60%')
    )

    output = widgets.Output()

    def plot_feature_graph(feature, neighbor_distance):
        output.clear_output(wait=True)
        with output:
            x = adata.obs["x"].values
            y = adata.obs["y"].values
            intensities = np.array(adata[:, feature].X).flatten()

            # Top N pixels
            top_indices = np.argsort(intensities)[-top_n:]
            coords = np.column_stack([x[top_indices], y[top_indices]])
            top_values = intensities[top_indices]

            # KDTree for neighbors
            tree = KDTree(coords)
            pairs = tree.query_pairs(neighbor_distance)

            edges = []
            for i, j in pairs:
                dist = np.linalg.norm(coords[i] - coords[j])
                edges.append((i, j, dist))

            degrees = np.zeros(len(coords))
            for i, j, _ in edges:
                degrees[i] += 1
                degrees[j] += 1

            # Build heatmap grid
            xi = np.unique(x)
            yi = np.unique(y)
            xi.sort()
            yi.sort()
            heatmap_matrix = np.full((len(yi), len(xi)), np.nan)

            x_idx = {val:i for i,val in enumerate(xi)}
            y_idx = {val:i for i,val in enumerate(yi)}
            for xi_val, yi_val, val in zip(x, y, intensities):
                heatmap_matrix[y_idx[yi_val], x_idx[xi_val]] = val

            # Plot
            fig = go.Figure()

            # Heatmap background
            fig.add_trace(go.Heatmap(
                z=heatmap_matrix,
                x=xi,
                y=yi,
                colorscale='Viridis',
                zmin=np.nanmin(intensities),
                zmax=np.nanmax(intensities),
                showscale=False,
                opacity=0.6
            ))

            # Edges
            for i, j, dist in edges:
                fig.add_trace(go.Scatter(
                    x=[coords[i,0], coords[j,0]],
                    y=[coords[i,1], coords[j,1]],
                    mode='lines',
                    line=dict(width=max(0.5, 4 - dist / neighbor_distance * 3), color='gray'),
                    hoverinfo='none'
                ))

            # Nodes
            fig.add_trace(go.Scatter(
                x=coords[:,0],
                y=coords[:,1],
                mode='markers',
                marker=dict(
                    size=5 + 3 * degrees / np.max(degrees) if np.max(degrees) > 0 else 5,
                    color=top_values,
                    colorscale='Viridis',
                    showscale=True,
                    colorbar=dict(title="Intensity")
                ),
                text=[f"Intensity: {v:.4f}<br>Degree: {int(d)}" for v, d in zip(top_values, degrees)],
                hoverinfo='text',
                name='Top pixels'
            ))

            fig.update_layout(
                title=f"Feature: {feature} — Top {top_n} Pixels with Heatmap",
                xaxis=dict(title='X (μm)', scaleanchor='y', scaleratio=1, autorange='reversed'),
                yaxis=dict(title='Y (μm)', autorange='reversed'),
                height=700,
                template='plotly_white'
            )

            fig.show()

    widgets.interactive_output(
        plot_feature_graph,
        {'feature': feature_dropdown, 'neighbor_distance': neighbor_slider}
    )

    display(widgets.VBox([feature_dropdown, neighbor_slider, output]))


In [1]:
# interactive graph with top 50 pixels and neighborhood = 3 pixels
interactive_feature_graph_with_heatmap(adata, pixel_size=60, top_n=100, neighbor_distance=100)



NameError: name 'interactive_feature_graph_with_heatmap' is not defined